In [ ]:
!pip install -q scanpy leidenalg celltypist scvi-tools scikit-misc

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import anndata as ad
import leidenalg
import scipy.sparse as sp
import scvi
import celltypist

In [ ]:
import subprocess
files_dir = '/content/covid_pbmc'

links = ['https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557327/suppl/GSM4557327%5F555%5F1%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557328/suppl/GSM4557328%5F555%5F2%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557329/suppl/GSM4557329%5F556%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557330/suppl/GSM4557330%5F557%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557331/suppl/GSM4557331%5F558%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557332/suppl/GSM4557332%5F559%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557333/suppl/GSM4557333%5F561%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557334/suppl/GSM4557334%5FHIP002%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557335/suppl/GSM4557335%5FHIP015%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557336/suppl/GSM4557336%5FHIP023%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557337/suppl/GSM4557337%5FHIP043%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557338/suppl/GSM4557338%5FHIP044%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557339/suppl/GSM4557339%5FHIP045%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz']

#downloading the files using wget and decompressing them
for link in links:
  subprocess.run(['wget', '-P', files_dir, link], check = True)
  subprocess.run(f'gunzip {files_dir}/*.gz', shell = True, check = True)


In [ ]:
#Because the files are rds files i will use these libraries to convert to an anndata object
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects import r
from pathlib import Path

folder_path = Path('/content/covid_pbmc')
pandas2ri.activate()
readRDS = ro.r['readRDS']
adata_dict = {}

#dictionary for donor labels
donor_dict = {"GSM4557330": "covid_557",
              "GSM4557339": "HIP045",
              "GSM4557327": "covid_555_1",
              "GSM4557331": "covid_558",
              "GSM4557336": "HIP023",
              "GSM4557335": "HIP015",
              "GSM4557329": "covid_557",
              "GSM4557328": "covid_555_2",
              "GSM4557338": "HIP044",
              "GSM4557334": "HIP002",
              "GSM4557337": "HIP043",
              "GSM4557333": "covid_561",
              "GSM4557332": "covid_559",
               }

for item in folder_path.iterdir():
  obj = readRDS(str(item.resolve())) #returns the path of the file as a string which readrds expects
  exon = obj.rx2("exon")#the rds file contains 3 matrices exon, introns, and spanning, exon is the standard gene-expression matrix

  # convert R sparse matrix → SciPy sparse
  i = np.array(exon.slots["i"])
  p = np.array(exon.slots["p"])
  x = np.array(exon.slots["x"])
  dims = tuple(exon.slots["Dim"])

  X = sp.csc_matrix((x, i, p), shape=dims).T

  # build AnnData
  adata = ad.AnnData(X)
  adata.var_names = list(r['rownames'](exon))
  adata.obs_names = list(r['colnames'](exon))

  #extracts unique sample id and assigns it to a col
  sample_id = item.name.split('_')[0]
  adata.obs['sample'] = sample_id

  #extracts condition of the sample and assigns it to a col
  healthy_sample = item.name.split('_')[1]
  if healthy_sample.startswith('HIP'):
    adata.obs['condition'] = 'HEALTHY'
  else:
    adata.obs['condition'] = 'COVID'

  #adds a col for the donor label
  adata.obs['donor'] = adata.obs['sample'].map(donor_dict)

  #store the adata object back in the dictionary
  adata_dict[sample_id] = adata

In [ ]:
for sample_id, adata in adata_dict.items():

  #saves a layer of the raw data
  adata.layers['raw_data'] = adata.X.copy()

  #light qc filtering
  print(f'Before light qc filter of {sample_id}: {adata.X.shape}')
  sc.pp.filter_cells(adata, min_genes = 200)
  sc.pp.filter_genes(adata, min_cells = 10)
  print(f'After light qc filter of {sample_id}: {adata.X.shape}')
  print('  ')

  #qc metrics
  adata.var['mt'] = adata.var.index.str.startswith('MT-')
  adata.var['rb'] = adata.var.index.str.startswith(("RPS", "RPL","FAU", "MRPL", "RSL"))
  adata.var["hb"] = adata.var.index.str.startswith(r"^HB[ABDEGMQZ]\d*(?!\w)")
  sc.pp.calculate_qc_metrics(adata, qc_vars = ['mt', 'rb', 'hb'], percent_top = [20], log1p = True, inplace = True)

  #visualizing the data
  p1 = sns.displot(adata.obs['total_counts'], bins = 100, kde = False)
  p2 = sc.pl.violin(adata, 'pct_counts_mt')
  p3 = sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color = 'pct_counts_mt')

In [ ]:
from scipy.stats import median_abs_deviation

def is_outlier(adata, metric: str, nmads: int):
  M = adata.obs[metric]
  outlier = (M < np.median(M) - (nmads * median_abs_deviation(M))) | (M > np.median(M) + (nmads * median_abs_deviation(M)))
  return outlier

def is_mt_outlier(adata, metric: str, nmads: int):
  M = adata.obs[metric]
  outlier = (M > np.median(M) + (nmads * median_abs_deviation(M)))
  return outlier

In [ ]:
#since i do not have the raw counts matrices for this data i cannot
#use soupx for ambient rna removal, there are other tools i could use like decontx
#but due to computational limitations that isn't feasible
#so i will be using slightly stricter filtering thresholds for my qc

for sample_id, adata in adata_dict.items():
  adata.obs['outlier'] = (is_outlier(adata, 'log1p_total_counts', 4)
                          | is_outlier(adata, 'log1p_n_genes_by_counts', 4)
                          | is_outlier(adata, 'pct_counts_in_top_20_genes', 4))

  adata.obs["mt_outlier"] = is_mt_outlier(adata, "pct_counts_mt", 3)
  print(f'Total number of cells before removing outliers from {sample_id}: {adata.X.shape}')
  adata = adata[(~adata.obs['outlier']) & (~adata.obs['mt_outlier'])]
  print(f'Total number of cells after removing outliers from {sample_id}: {adata.X.shape}')
  print('  ')

  #visualizing the filtered data
  p1 = sns.displot(adata.obs['total_counts'], bins = 100, kde = False)
  p2 = sc.pl.violin(adata, 'pct_counts_mt')
  p3 = sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color = 'pct_counts_mt')

  #storing the filtered data
  adata.layers['qc_filtered'] = adata.X.copy()
  adata_dict[sample_id] = adata

In [ ]:
for adata in adata_dict.values():
  #normalize
  sc.pp.normalize_total(adata)
  sc.pp.log1p(adata)

  #select hvg
  sc.pp.highly_variable_genes(adata, n_top_genes = 3000, flavor = 'seurat_v3', subset = False)

#concat all the samples into one adata object
adata_all = sc.concat(adata_dict.values(), label = 'batch', keys = list(adata_dict.keys()))

#view the concatenated adata obj before integration
sc.tl.pca(adata_all)
sc.pp.neighbors(adata_all, n_neighbors = 30, n_pcs = 50)
sc.tl.umap(adata_all)
sc.pl.umap(adata_all, color = ['batch', 'condition'], wspace = 0.5)


In [ ]:
#scvi and scanvi work better when provided with labeled cell types,
#so i will rough labels with celltypist

temp_adata = adata_all.copy()
temp_adata.X = temp_adata.layers['qc_filtered']

temp_adata.X = temp_adata.X.astype('float32')
temp_adata.layers['qc_filtered'] = temp_adata.layers['qc_filtered'].astype('float32')
temp_adata.layers['raw_data'] = temp_adata.layers['raw_data'].astype('float32')
from scipy.sparse import csr_matrix
temp_adata.X = csr_matrix(temp_adata.X)

sc.pp.normalize_total(temp_adata, target_sum = 1e4)
sc.pp.log1p(temp_adata)

celltypist.models.download_models(force_update=True, model=["Immune_All_High.pkl"])
model = celltypist.models.Model.load("Immune_All_High.pkl")
prediction = celltypist.annotate(temp_adata, model = model)

adata_all.obs['rough_cell_type_label'] = prediction.predicted_labels
del temp_adata

sc.pl.umap(adata_all, color = ['batch', 'rough_cell_type_label'], wspace = 1)

In [ ]:
#doing batch correction and integration with scvi and scanvi

#trains the initial scvi model
scvi.model.SCVI.setup_anndata(adata_all,
                              layer = 'raw_data',
                              batch_key = 'batch')
scvi_model = scvi.model.SCVI(adata_all)
scvi_model.train()
adata_all.obsm['X_scvi'] = scvi_model.get_latent_representation()

#trains the scanvi model based on the previous scvi model
scanvi_model = scvi.model.SCANVI.from_scvi_model(scvi_model,
                                                 labels_key = 'rough_cell_type_label',
                                                 unlabeled_category = 'unknown')
scanvi_model.train()
adata_all.obsm['X_scanvi'] = scanvi_model.get_latent_representation()

In [ ]:
#saving a copy of the integrated adata_all file
adata_all.write_h5ad('adata_all_integrated', compression = 'gzip')

#visualizing the integrated adata_all object
sc.pp.neighbors(adata_all, use_rep='X_scvi')
sc.tl.umap(adata_all)
sc.pl.umap(adata_all, color = ['rough_cell_type_label', 'batch'], wspace = 1)

sc.pp.neighbors(adata_all, use_rep='X_scanvi')
sc.tl.umap(adata_all)
sc.pl.umap(adata_all, color = ['rough_cell_type_label', 'batch'], wspace = 1)